# Chapter 11: Memory in Agents + MCP
## Topic 5: Connecting an Agent to MCP End to End

**One notebook. Theory + Code together.**


## Part A: Theory

### 1. Concept, Intuition, and Why This Exists

- Topic 4 built and verified a real MCP *server* — but only tested it by calling its tool directly through the server object's own `call_tool()` method, from within the same Python process. This topic completes the actual, full loop: a genuine, separate MCP *client* — the piece of code an agent would actually use — connecting to that server, discovering its tools, and invoking them, exactly the client-server relationship Topic 3 described conceptually and Topic 4 only partially exercised.
- The core distinction this topic draws precisely: Topic 4 proved the *server* works correctly. This topic proves the full *protocol conversation* works correctly — a client that has no hardcoded, advance knowledge of what tools this specific server offers can still discover them (via `list_tools()`) and correctly invoke them (via the client's own `call_tool()`), exactly the standardized-discovery mechanism that makes MCP's N+M integration model (Topic 3) genuinely work in practice, not just in the cost arithmetic.
- Why this matters as this chapter's closing exercise: this is the point where Chapter 11's entire arc — short-term vs long-term memory (Topic 1), a genuine persistent store (Topic 2), MCP's conceptual value (Topic 3), a real server (Topic 4) — comes together into one complete, working, end-to-end demonstration: an agent-like client genuinely retrieving long-term memory through a standardized protocol, rather than any single piece being demonstrated in isolation.


### 2. Internal Working — Step by Step

**The full client-server MCP interaction, step by step, using the real SDK:**

1. **Establish a connection between a client and the server** — in a genuine deployment, this would happen over a real transport (stdio for a local subprocess, or HTTP/SSE for a network-accessible server); this topic's exercise uses the SDK's own in-memory connection utility (`create_connected_server_and_client_session`), which exercises the exact same client-server protocol logic without needing to actually spawn a separate process or open a real network connection — appropriate for this notebook's demonstration purposes, though a genuine production deployment would use a real transport.
2. **The client calls `list_tools()`** — this is the client-side half of Topic 4's server-side `list_tools()` call: the client sends a real protocol request, the server responds with the schema it automatically derived (Topic 4), and the client receives this information *without* having had any of it hardcoded in advance — this is the concrete mechanism underlying Topic 3's claim that a standardized protocol enables genuine tool discovery, not just tool description.
3. **The client constructs and sends a `call_tool()` request** — mirroring exactly the same request/dispatch/result mechanics Chapter 10 Topic 2 established for this project's own internal agent loop, but now happening across a genuine (if in-memory, for this exercise) client-server boundary rather than within a single Python program's own function calls.
4. **The result flows back to the client**, completing the loop — at this point, an agent using this MCP client would take this result and inject it into its own conversation's short-term memory (Chapter 11 Topic 1's short-term memory concept), exactly as this project's actual `run_agent()` currently does with `validate_fd_reference`'s direct-call result, just now sourced through a standardized protocol instead of a direct, in-process function call.


### 3. How This Is Implemented in This Project

- This topic's exercise closes Chapter 11's full arc concretely: the same `check_sender_history` tool built in Topic 2, wrapped in the same MCP server built in Topic 4, is now called by a genuine, separate MCP client — demonstrating the complete, standardized alternative to this project's current, correct direct-integration approach (Chapter 10), without this notebook claiming that alternative should actually replace the current approach yet, per Topic 3's own concrete-trigger conclusion.
- A genuinely production-adopted version of this pattern — if and when Topic 3's triggers were actually met — would have this project's `run_agent()` loop use an MCP client (instead of, or alongside, its current direct dispatch logic) to discover and call tools, meaning the *agent's own reasoning process* (deciding whether and when to call `check_sender_history`, exactly per Chapter 9 Topic 1's triggering logic and Chapter 10 Topic 6's tool-selection discussion) would remain unchanged — only the underlying *mechanism* by which a decided-upon tool call actually gets executed would differ.
- This exercise's honest scope: it demonstrates the genuine MCP protocol mechanics working correctly end to end, using the real SDK — it does not build a full, LLM-driven agent that autonomously decides to call this MCP-wrapped tool as part of its own reasoning (that would require integrating this MCP client into an actual `run_agent()`-style loop with a real or simulated model call, a larger undertaking than this closing topic's scope, though a natural extension of everything this chapter has built).


### 4. Real-World Issues, Edge Cases, Debugging, Monitoring, Scaling, Latency, Cost, Security, Deployment

- **The in-memory connection used in this exercise is a genuine simplification worth being explicit about.** It correctly exercises the real protocol's request/response logic, but a genuine production deployment would need a real transport — stdio for a local, subprocess-based server, or HTTP/SSE for a network-accessible one — each with its own connection-management, retry, and failure-handling concerns beyond what this in-memory exercise needed to address.
- **Client-side error handling needs the same rigor Chapter 10 Topic 5 established for this project's own direct tool calls** — a real MCP client, calling a real (potentially remote, potentially unreliable) server, needs its own transient-vs-permanent error distinction and retry logic, exactly mirroring Chapter 10 Topic 5's principles, now applied at a genuine client-server protocol boundary rather than within trusted, single-process code.
- **Latency for a genuinely networked MCP connection is a real, additional cost compared to this project's current direct, in-process function calls** — Topic 3 already flagged this trade-off conceptually; this topic's exercise, using an in-memory connection, doesn't itself measure real network latency, but a genuine deployment decision would need to measure and account for it directly, not assume it away.
- **Debugging a failure in this full client-server loop requires checking three distinct places**, extending Topic 4's two-layer debugging guidance: the underlying tool function's own logic (Topic 2), the server's protocol-handling layer (Topic 4), and now also the client's own connection and request-construction logic — three genuinely separate potential failure points, each needing its own isolated verification rather than assuming a failure automatically belongs to whichever layer seems most likely without actually checking.
- **Monitoring a genuinely deployed client-server MCP setup** would need visibility into both sides of the connection — request/response timing and error rates from the client's perspective, and tool-invocation counts and latency from the server's perspective — a more distributed monitoring concern than this project's current single-process, direct-integration approach requires.


### 5. Design Decisions, Trade-offs, and Real-Time Dilemmas

- **In-memory connection (this exercise's approach) vs a real transport for testing:** the in-memory utility is the right choice for verifying protocol *logic* correctness quickly and simply, exactly this topic's purpose — a genuine pre-production validation would still need to test over a real transport at least once, since transport-level concerns (network reliability, serialization over an actual wire format) aren't exercised by an in-memory connection.
- **How much of this project's own agent logic should eventually move to use an MCP client, if Topic 3's triggers were ever met:** a full migration (every tool call routed through MCP) vs a partial one (only tools genuinely shared with other consumers move to MCP, while project-specific tools stay directly integrated) — the partial approach avoids paying MCP's overhead for tools that never actually need cross-consumer sharing, a more targeted, evidence-based migration than an all-or-nothing switch.
- **Whether the agent's own tool-selection reasoning changes at all when tools are accessed via MCP instead of direct integration:** the theory's own conclusion is that it shouldn't — Chapter 9 Topic 1's triggering logic and Chapter 10 Topic 6's tool-selection logic are about *deciding what to call*, entirely separate from *how the call actually gets executed* — but this separation needs to be verified in practice, not just assumed, if this pattern were ever genuinely adopted in production.


### 6. Alternatives and When to Use Each

- **In-memory client-server connection (this topic's approach for verification):** the right choice for quickly and correctly validating protocol logic during development, exactly this closing exercise's purpose.
- **A real stdio or HTTP-based transport:** necessary for any genuine production deployment, and worth testing explicitly before considering an MCP-based approach production-ready, since transport-level concerns aren't exercised by the in-memory alternative.
- **Continuing with direct integration (this project's actual current, correct production approach):** remains the right choice until Topic 3's concrete adoption triggers are genuinely met — this entire chapter's MCP exercises (Topics 3-5) are readiness and learning work, not a call to change the current production architecture prematurely.


### 7. Common Mistakes and Production Failures

- Treating a successful in-memory client-server test as equivalent to production-readiness over a genuine network transport, without separately validating the transport-level concerns an in-memory connection doesn't exercise.
- Not applying Chapter 10 Topic 5's error-handling and retry discipline to the client side of a genuine MCP connection, assuming a remote server call is as reliable as this project's current in-process, direct function calls.
- Assuming an agent's tool-selection reasoning needs to change once tools are accessed via MCP rather than direct integration, rather than correctly recognizing that tool selection and tool execution mechanism are genuinely separate concerns.
- Migrating this project's entire tool set to MCP all at once, rather than a more targeted, evidence-based partial migration limited to tools that genuinely have a second consumer or other concrete trigger justifying the change.
- Not maintaining clear separation between this chapter's honest "learning and readiness exercise" framing and an actual recommendation to change this project's current, correct production architecture — conflating the two risks a premature, unjustified adoption decision.


### 8. Lead-Level Interview Questions

**Basic**

- Q: What does this topic's exercise demonstrate that Topic 4's server-only demonstration didn't?
  A: Topic 4 verified the server works correctly by calling its tool directly through the server object's own method, within the same process. This topic adds a genuine, separate client that discovers the server's tools via a real protocol request (`list_tools()`) and invokes them via another real protocol request (`call_tool()`) — proving the full client-server protocol conversation works, not just the server's own internal logic.

- Q: Why does this exercise use an in-memory connection rather than a real network transport?
  A: The in-memory connection utility exercises the genuine protocol request/response logic correctly and quickly, appropriate for verifying correctness during development — a real transport (stdio or HTTP/SSE) would additionally exercise transport-level concerns (network reliability, actual wire-format serialization) that aren't relevant to validating the protocol logic itself, which is this exercise's specific purpose.

**Intermediate**

- Q: If this project's agent eventually used an MCP client instead of direct tool integration, would its tool-selection logic (deciding whether and when to call a specific tool) need to change?
  A: No — tool selection (Chapter 9 Topic 1's triggering logic, Chapter 10 Topic 6's multi-tool selection logic) is about deciding *what* to call, which is entirely separate from *how* a decided-upon call actually gets executed. An MCP client changes the execution mechanism, not the reasoning that decides when a tool call is appropriate in the first place — though this separation should be verified in practice, not just assumed, before treating it as settled.

- Q: What are the three distinct places a failure in this full client-server loop could originate, and why does distinguishing them matter?
  A: The underlying tool function's own logic (Topic 2's `check_sender_history` implementation), the server's protocol-handling layer (Topic 4's `FastMCP` wrapping), and the client's own connection and request-construction logic (this topic's addition). Each requires a genuinely different fix, and correctly attributing a failure to the right layer — rather than guessing based on which seems most likely — is what makes debugging efficient rather than wasted effort in the wrong place.

**Advanced**

- Q: Design a phased migration plan for moving this project's tools from direct integration to MCP, assuming Topic 3's adoption triggers are eventually met.
  A: Start with the specific tool that actually has the concrete trigger (for example, a genuine second consumer needing `check_sender_history` specifically), migrating only that tool to MCP while leaving the rest of the tool set on direct integration — validating the pattern works in a real, production, networked deployment (not just this topic's in-memory exercise) before considering broader migration. Tools without a genuine second consumer or other concrete trigger should remain on direct integration, since migrating them would add MCP's real overhead (Topic 3) without a corresponding benefit — an evidence-based, tool-by-tool migration rather than an all-or-nothing switch.

- Q: A production incident shows an MCP-based tool call failed. How would you determine whether this is a client-side, server-side, or underlying-function issue?
  A: Check the underlying function directly first (call `check_sender_history_impl()` — Topic 4's approach — bypassing MCP entirely) to confirm whether the core logic itself is working correctly. If it is, check the server's own tool-invocation logs (Topic 4's layer) to see whether the request reached the server correctly and was processed without error. If the server-side logs show a clean, successful execution but the client never received a correct result, the issue is most likely in the client's own connection handling or the transport layer between them — this systematic, layer-by-layer elimination is what makes a three-layer failure genuinely diagnosable rather than a guessing exercise.

**Scenario-based**

- Q: A stakeholder, having seen this chapter's working MCP demonstrations, asks why the project hasn't yet switched its production agent to use MCP for all its tools. How do you explain this clearly?
  A: This chapter's exercises (Topics 3-5) deliberately demonstrate that the *mechanism* works correctly and that this project's tools are well-designed enough to require no rework for MCP-compatibility — genuinely valuable readiness work. But MCP's real benefit (standardized, reusable tool access across multiple consumers) doesn't yet apply at this project's actual current scale, which has exactly one agent using its own tools. Adopting MCP now would add real communication overhead and a new operational surface (a genuinely deployed, monitored server) without a corresponding benefit — the project is deliberately waiting for a concrete trigger (a genuine second consumer, a process-isolation need, cross-team reuse) before that trade-off becomes worthwhile, exactly the same evidence-based discipline already applied to the vector database migration (Chapter 6) and the sender-history store's own persistence choice (Chapter 11 Topic 2).

**Follow-up questions to expect:**

- "What would the first real, non-demonstration MCP deployment for this project actually look like?"
- "How would you test this same client-server pattern over a real network transport, rather than in-memory?"


### 9. Hidden Concepts / Prerequisites Worth Knowing

- **This topic's in-memory connection technique is itself a specific instance of a broader software testing pattern: testing protocol or interface logic without incurring the cost and flakiness of a genuine network dependency**, directly analogous to Chapter 10 Topic 5's and Topic 7's use of simulated, controllable failure infrastructure instead of real API calls for testing application logic — the same underlying testing philosophy (isolate what you're actually verifying from dependencies you don't need to exercise for that specific test) applied to a different layer of the system.
- **The clean separation between "deciding what to call" (tool selection) and "how the call executes" (direct integration vs MCP) is a specific instance of a general software architecture principle: separating policy from mechanism** — the decision logic and the execution mechanism can vary independently, a recurring, valuable pattern in well-architected systems well beyond this specific agent-and-tools context.
- **This topic closes Chapter 11's complete arc, and sets up a natural connection to Chapter 12's orchestration discussion:** as this project's tool-composition and tool-selection logic grows more complex (multi-tool agents, Chapter 10 Topic 6; long-term memory retrieval, Chapter 11 Topics 1-2; potentially MCP-mediated execution, this chapter's Topics 3-5), a more structured orchestration framework — exactly Chapter 12's subject — becomes an increasingly natural next step for managing that growing complexity cleanly.

### 10. Quick Revision Summary (for last-minute interview prep)

> This closing exercise completes Chapter 11's full arc: a genuine MCP client, using the real SDK's in-memory connection utility, discovers Topic 4's server's tools via a real `list_tools()` protocol request and invokes `check_sender_history` via a real `call_tool()` protocol request — proving the complete client-server protocol conversation works correctly, not just the server's own internal logic (which Topic 4 alone verified). This demonstrates that a decided-upon tool call's *execution mechanism* (direct integration vs MCP) is a genuinely separate concern from the *decision* of whether and when to call that tool (Chapter 9 Topic 1's triggering logic, Chapter 10 Topic 6's multi-tool selection) — an agent's reasoning process shouldn't need to change based on how a chosen tool call actually gets executed underneath. A genuine production deployment would need a real transport (not this exercise's in-memory shortcut), client-side error handling mirroring Chapter 10 Topic 5's principles, and three-layer debugging discipline (underlying function, server, client) rather than guessing which layer a failure belongs to. This entire chapter's MCP work remains readiness and learning exercise, not a production recommendation — Topic 3's concrete adoption triggers still need to be genuinely met before this project's actual, currently-correct direct-integration architecture should change.


### Module 1: The Real Server From Topic 4, Ready to Be Connected To

Rebuild Topic 4's real MCP server (same tool, same implementation) as the target for this topic's genuine client connection.

In [1]:

# ------------------------------------------------------------------
# MODULE 1: The real server, ready for a genuine client connection
# ------------------------------------------------------------------

import csv
import os
from datetime import datetime
from mcp.server.fastmcp import FastMCP

SENDER_HISTORY_PATH = "/home/claude/build/sender_history_e2e_demo.csv"
FIELDNAMES = ["sender_email", "first_seen_date", "total_email_count",
              "last_topic", "has_unresolved_issue", "last_contact_date"]

def _read_all_records() -> dict:
    if not os.path.exists(SENDER_HISTORY_PATH):
        return {}
    records = {}
    with open(SENDER_HISTORY_PATH, newline="", encoding="utf-8") as f:
        reader = csv.DictReader(f)
        for row in reader:
            row["total_email_count"] = int(row["total_email_count"])
            row["has_unresolved_issue"] = row["has_unresolved_issue"] == "True"
            records[row["sender_email"]] = row
    return records

def _write_all_records(records: dict):
    with open(SENDER_HISTORY_PATH, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=FIELDNAMES)
        writer.writeheader()
        for record in records.values():
            writer.writerow(record)

def check_sender_history_impl(sender_email: str) -> dict:
    records = _read_all_records()
    record = records.get(sender_email.strip().lower())
    if record is None:
        return {"sender_email": sender_email, "is_repeat_sender": False}
    return {"sender_email": sender_email, "is_repeat_sender": True, **record}

def write_sender_interaction_impl(sender_email: str, topic: str, unresolved: bool):
    key = sender_email.strip().lower()
    records = _read_all_records()
    today = datetime.now().strftime("%Y-%m-%d")
    if key not in records:
        records[key] = {"sender_email": key, "first_seen_date": today, "total_email_count": 1,
                          "last_topic": topic, "has_unresolved_issue": unresolved, "last_contact_date": today}
    else:
        existing = records[key]
        existing["total_email_count"] += 1
        existing["last_topic"] = topic
        existing["has_unresolved_issue"] = unresolved
        existing["last_contact_date"] = today
    _write_all_records(records)

if os.path.exists(SENDER_HISTORY_PATH):
    os.remove(SENDER_HISTORY_PATH)
write_sender_interaction_impl("shobha.chopra@email.com", "premature withdrawal penalty", unresolved=True)

mcp_server = FastMCP("sender-history-server")

@mcp_server.tool()
def check_sender_history(sender_email: str) -> dict:
    """Look up whether this sender has contacted us before, and if so,
    what their last topic was and whether it was resolved.
    Call this at the start of handling a new email to get context
    about the sender's history with us."""
    return check_sender_history_impl(sender_email)

print("=" * 70)
print("MODULE 1: SERVER READY (identical pattern to Topic 4)")
print("=" * 70)
print(f"Server '{mcp_server.name}' ready, with 1 real tool registered.")
print("\nModule 1 complete. Run Module 2 next.")


MODULE 1: SERVER READY (identical pattern to Topic 4)
Server 'sender-history-server' ready, with 1 real tool registered.

Module 1 complete. Run Module 2 next.


### Module 2: A Genuine, Separate MCP Client — Real End-to-End Connection

Use the SDK's real in-memory client-server utility to establish an actual protocol connection, then have the CLIENT (not the server object directly) discover and call the tool.

In [2]:

# ------------------------------------------------------------------
# MODULE 2: A GENUINE, SEPARATE client -- real end-to-end connection
# ------------------------------------------------------------------

import asyncio
from mcp.shared.memory import create_connected_server_and_client_session

async def run_end_to_end_demo():
    """Establishes a REAL client-server connection (in-memory transport,
    but REAL protocol logic) and drives the FULL interaction from the
    CLIENT's side -- exactly what an agent's own MCP client would do."""
    async with create_connected_server_and_client_session(mcp_server._mcp_server) as client_session:

        # STEP 1: the client discovers tools -- NO hardcoded advance
        # knowledge of what this server offers
        discovered = await client_session.list_tools()
        print("STEP 1 -- Client discovers tools via REAL list_tools() request:")
        for tool in discovered.tools:
            print(f"  Discovered tool: '{tool.name}'")
            print(f"  Description: {tool.description[:80]}...")

        # STEP 2: the client constructs and sends a REAL call_tool request
        print("\nSTEP 2 -- Client sends a REAL call_tool() request:")
        response = await client_session.call_tool(
            "check_sender_history", {"sender_email": "shobha.chopra@email.com"}
        )
        print(f"  Response received by client: {response.content}")

        # STEP 3: a genuinely first-time sender, through the SAME client
        print("\nSTEP 3 -- Same client, a genuinely first-time sender:")
        response_new = await client_session.call_tool(
            "check_sender_history", {"sender_email": "brand.new.customer@email.com"}
        )
        print(f"  Response received by client: {response_new.content}")

        return discovered, response, response_new


print("=" * 70)
print("MODULE 2: REAL END-TO-END CLIENT-SERVER MCP INTERACTION")
print("=" * 70)
print()

discovered_tools, first_response, second_response = asyncio.run(run_end_to_end_demo())

print("\nThis is a GENUINE client-server protocol conversation -- the client")
print("never had check_sender_history's schema hardcoded anywhere in its")
print("own code. It learned about the tool ENTIRELY through the real")
print("list_tools() protocol call, then correctly invoked it via call_tool().")
print("\nModule 2 complete. Run Module 3 next.")


MODULE 2: REAL END-TO-END CLIENT-SERVER MCP INTERACTION

STEP 1 -- Client discovers tools via REAL list_tools() request:
  Discovered tool: 'check_sender_history'
  Description: Look up whether this sender has contacted us before, and if so,
    what their l...

STEP 2 -- Client sends a REAL call_tool() request:
  Response received by client: [TextContent(type='text', text='{\n  "sender_email": "shobha.chopra@email.com",\n  "is_repeat_sender": true,\n  "first_seen_date": "2026-07-09",\n  "total_email_count": 1,\n  "last_topic": "premature withdrawal penalty",\n  "has_unresolved_issue": true,\n  "last_contact_date": "2026-07-09"\n}', annotations=None, meta=None)]

STEP 3 -- Same client, a genuinely first-time sender:
  Response received by client: [TextContent(type='text', text='{\n  "sender_email": "brand.new.customer@email.com",\n  "is_repeat_sender": false\n}', annotations=None, meta=None)]

This is a GENUINE client-server protocol conversation -- the client
never had check_sender_hi

### Module 3: Simulating How an Agent Would Actually Use This Client

Wrap the real MCP client call in an agent-shaped function, showing exactly how this fits into the existing short-term memory (Topic 1) pattern -- completing this chapter's full arc.

In [3]:

# ------------------------------------------------------------------
# MODULE 3: How an agent would actually USE this real MCP client
# ------------------------------------------------------------------

async def agent_turn_with_mcp_memory(sender_email: str, email_subject: str, email_content: str) -> dict:
    """Simulates what run_agent() would do with an MCP-based tool --
    builds SHORT-TERM memory (Chapter 11 Topic 1's messages list)
    using a result retrieved via the REAL MCP client, completing the
    full loop this chapter has built piece by piece."""
    async with create_connected_server_and_client_session(mcp_server._mcp_server) as client_session:
        # the agent's OWN reasoning would decide to call this tool --
        # exactly Chapter 9 Topic 1 / Chapter 10 Topic 6's triggering logic,
        # UNCHANGED regardless of execution mechanism (this topic's key claim)
        mcp_response = await client_session.call_tool(
            "check_sender_history", {"sender_email": sender_email}
        )

        # inject the REAL MCP-retrieved result into SHORT-TERM memory
        # (Chapter 11 Topic 1's messages list), exactly as this project's
        # actual run_agent() does with validate_fd_reference's direct result
        messages = [{"role": "user", "content": f"Subject: {email_subject}\n\nBody: {email_content}"}]
        messages.insert(0, {"role": "user", "content": f"[Sender history via MCP: {mcp_response.content}]"})

        return {"messages": messages}


print("=" * 70)
print("MODULE 3: COMPLETING THE FULL LOOP -- MCP RESULT INTO SHORT-TERM MEMORY")
print("=" * 70)

agent_result = asyncio.run(agent_turn_with_mcp_memory(
    "shobha.chopra@email.com", "Follow-up", "Any update on my earlier question?"
))

print("\nFinal messages list an agent would send to the model:")
for m in agent_result["messages"]:
    content_preview = str(m["content"])[:100]
    role_val = m["role"]
    print(f"  [{role_val}] {content_preview}...")

print("\nCONFIRMED: this completes Chapter 11's full arc --")
print("  Topic 1: short-term (messages list) vs long-term memory concepts")
print("  Topic 2: a real, tested persistent long-term memory store")
print("  Topic 3: WHY a standardized protocol (MCP) matters, and when")
print("  Topic 4: a REAL MCP server wrapping that real store")
print("  Topic 5: a REAL client connecting end-to-end, feeding the result")
print("           into short-term memory -- exactly Topic 1's concept,")
print("           now demonstrated working through the FULL real stack.")

print("\nModule 3 complete. All Chapter 11 Topic 5 modules done.")
print()
print("=" * 70)
print("CHAPTER 11 TOPIC 5 -- KEY POINTS TO REMEMBER")
print("=" * 70)
print("""
  A GENUINE, separate client -- not just calling the server object
  directly (Topic 4) -- discovers tools via REAL list_tools() and
  invokes them via REAL call_tool(), using the official SDK throughout.

  Tool SELECTION logic (deciding whether/when to call a tool) is
  UNCHANGED by execution mechanism (direct integration vs MCP) --
  demonstrated concretely: the same short-term-memory-injection pattern
  from Topic 1 works identically regardless of how the result was
  actually retrieved underneath.

  Three-layer debugging: underlying function (Topic 2), server (Topic 4),
  client (this topic) -- each a genuinely separate potential failure
  point requiring separate verification.

  This chapter's MCP work (Topics 3-5) remains READINESS/LEARNING --
  Topic 3's concrete adoption triggers still need to be met before this
  replaces the project's current, correct direct-integration approach.
""")


MODULE 3: COMPLETING THE FULL LOOP -- MCP RESULT INTO SHORT-TERM MEMORY

Final messages list an agent would send to the model:
  [user] [Sender history via MCP: [TextContent(type='text', text='{\n  "sender_email": "shobha.chopra@email.c...
  [user] Subject: Follow-up

Body: Any update on my earlier question?...

CONFIRMED: this completes Chapter 11's full arc --
  Topic 1: short-term (messages list) vs long-term memory concepts
  Topic 2: a real, tested persistent long-term memory store
  Topic 3: WHY a standardized protocol (MCP) matters, and when
  Topic 4: a REAL MCP server wrapping that real store
  Topic 5: a REAL client connecting end-to-end, feeding the result
           into short-term memory -- exactly Topic 1's concept,
           now demonstrated working through the FULL real stack.

Module 3 complete. All Chapter 11 Topic 5 modules done.

CHAPTER 11 TOPIC 5 -- KEY POINTS TO REMEMBER

  A GENUINE, separate client -- not just calling the server object
  directly (Topic 4) -- 